In [1]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

import warnings
import os
import joblib
import json

warnings.filterwarnings('ignore')

os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv('../data/home_vehicle_loans.csv')

# -----------------------------
# Create Figure
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fig.suptitle(
    'Survival Analysis: Time to Loan Default',
    fontsize=13,
    fontweight='bold'
)

# -----------------------------
# Kaplan-Meier by LTV
# -----------------------------
kmf = KaplanMeierFitter()

ltv_groups = {
    'Low LTV (<70%)': df['ltv'] < 0.70,
    'Mid LTV (70-85%)': (
        (df['ltv'] >= 0.70) &
        (df['ltv'] < 0.85)
    ),
    'High LTV (85%+)': df['ltv'] >= 0.85
}

colors = ['#16A34A','#EAB308','#EF4444']

for (label, mask), color in zip(ltv_groups.items(), colors):

    grp = df[mask]

    kmf.fit(
        durations=grp['time_to_event'],
        event_observed=grp['event'],
        label=label
    )

    kmf.plot_survival_function(
        ax=axes[0],
        color=color,
        linewidth=2
    )

axes[0].set_title(
    'Survival Probability by LTV Group',
    fontweight='bold'
)

axes[0].set_xlabel('Months')

axes[0].set_ylabel('Survival Probability (No Default)')

axes[0].legend()

axes[0].axhline(
    0.9,
    color='gray',
    linestyle='--',
    alpha=0.5
)

# -----------------------------
# Kaplan-Meier by CIBIL
# -----------------------------
cibil_groups = {
    'Poor CIBIL (<650)': df['cibil'] < 650,
    'Fair CIBIL (650-750)': (
        (df['cibil'] >= 650) &
        (df['cibil'] < 750)
    ),
    'Good CIBIL (750+)': df['cibil'] >= 750
}

for (label, mask), color in zip(cibil_groups.items(), colors):

    grp = df[mask]

    kmf.fit(
        durations=grp['time_to_event'],
        event_observed=grp['event'],
        label=label
    )

    kmf.plot_survival_function(
        ax=axes[1],
        color=color,
        linewidth=2
    )

axes[1].set_title(
    'Survival Probability by CIBIL Group',
    fontweight='bold'
)

axes[1].set_xlabel('Months')

axes[1].set_ylabel('Survival Probability (No Default)')

axes[1].legend()

plt.tight_layout()

plt.savefig(
    '../plots/kaplan_meier.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/kaplan_meier.png")

# -----------------------------
# Log-rank Test
# -----------------------------
low = df[df['ltv'] < 0.70]

high = df[df['ltv'] >= 0.85]

result = logrank_test(
    low['time_to_event'],
    high['time_to_event'],
    low['event'],
    high['event']
)

print()
print("Log-rank test (Low LTV vs High LTV)")

print(f"p-value: {result.p_value:.6f}")

print(
    f"Significant: {'YES' if result.p_value < 0.05 else 'NO'}"
)

Saved: plots/kaplan_meier.png

Log-rank test (Low LTV vs High LTV)
p-value: 0.000000
Significant: YES


In [2]:
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Cox Features
# -----------------------------
cox_features = [
    'ltv',
    'emi_income',
    'cibil',
    'loan_amt',
    'tenure_mo',
    'interest_rate',
    'delinq_hist',
    'co_applicant',
    'age',
    'income_mo',
    'emp_years'
]

df_cox = df[
    cox_features +
    ['time_to_event','event']
].copy()

# -----------------------------
# Standard Scaling
# -----------------------------
scaler_cox = StandardScaler()

df_cox[cox_features] = scaler_cox.fit_transform(
    df_cox[cox_features]
)

# Sample for faster training
df_cox_sample = df_cox.sample(
    20000,
    random_state=42
).reset_index(drop=True)

print("Fitting Cox PH model...")

# -----------------------------
# Cox PH Model
# -----------------------------
cph = CoxPHFitter(
    penalizer=0.1
)

cph.fit(
    df_cox_sample,
    duration_col='time_to_event',
    event_col='event',
    show_progress=False
)

print()
print("Cox PH Model Summary")

print(
    cph.summary[
        ['coef','exp(coef)','p']
    ].round(4)
)

print()
print("HAZARD RATIOS")

hr_summary = cph.summary[
    'exp(coef)'
].sort_values(ascending=False)

for feat, hr in hr_summary.items():

    direction = (
        "INCREASES"
        if hr > 1.0
        else "DECREASES"
    )

    print(
        f"{feat:20}: HR={hr:.3f} ({direction} risk)"
    )

# -----------------------------
# C-index
# -----------------------------
c_index = cph.concordance_index_

print()
print(f"C-index: {c_index:.4f}")

# -----------------------------
# Hazard Ratio Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 7))

cph.plot(ax=ax)

ax.set_title(
    'Cox PH — Hazard Ratios with 95% CI',
    fontweight='bold'
)

ax.axvline(
    0,
    color='black',
    linewidth=1
)

plt.tight_layout()

plt.savefig(
    '../plots/cox_hazard_ratios.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/cox_hazard_ratios.png")

# -----------------------------
# Save Models
# -----------------------------
joblib.dump(
    cph,
    '../models/cox_model.pkl'
)

joblib.dump(
    scaler_cox,
    '../models/cox_scaler.pkl'
)

json.dump(
    {
        'c_index': round(c_index,4),
        'cox_features': cox_features,
        'n_train': len(df_cox_sample)
    },
    open('../models/cox_metrics.json','w'),
    indent=2
)

print("Saved: models/cox_model.pkl")

Fitting Cox PH model...

Cox PH Model Summary
                 coef  exp(coef)       p
covariate                               
ltv            0.2205     1.2467  0.0000
emi_income     0.1625     1.1765  0.0000
cibil         -0.2691     0.7640  0.0000
loan_amt      -0.1041     0.9011  0.0000
tenure_mo     -0.4081     0.6649  0.0000
interest_rate  0.1296     1.1383  0.0000
delinq_hist    0.1274     1.1359  0.0000
co_applicant  -0.0621     0.9398  0.0000
age           -0.0083     0.9918  0.5351
income_mo     -0.0644     0.9376  0.0000
emp_years      0.0130     1.0131  0.3282

HAZARD RATIOS
ltv                 : HR=1.247 (INCREASES risk)
emi_income          : HR=1.176 (INCREASES risk)
interest_rate       : HR=1.138 (INCREASES risk)
delinq_hist         : HR=1.136 (INCREASES risk)
emp_years           : HR=1.013 (INCREASES risk)
age                 : HR=0.992 (DECREASES risk)
co_applicant        : HR=0.940 (DECREASES risk)
income_mo           : HR=0.938 (DECREASES risk)
loan_amt            : 